In [ ]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [ ]:
import pandas as pd
df = pd.read_csv('/content/drive/MyDrive/1:1_Krish_Sathish/Data/Mirai_database.csv')
print(df.head())


   flow_duration  Header_Length  Protocol Type  Duration         Rate  \
0       0.000000            0.0          47.00     64.00    29.117203   
1       0.140556       142876.6          17.00     64.00  1862.255595   
2      52.760724       687278.6           8.70     76.40    24.574868   
3       0.000000            0.0          47.00     64.00    49.921493   
4       2.967012      5620219.2          16.35     69.09  3419.303567   

         Srate  Drate  fin_flag_number  syn_flag_number  rst_flag_number  ...  \
0    29.117203    0.0              0.0              0.0              0.0  ...   
1  1862.255595    0.0              0.0              0.0              0.0  ...   
2    24.574868    0.0              0.0              0.0              0.0  ...   
3    49.921493    0.0              0.0              0.0              0.0  ...   
4  3419.303567    0.0              0.0              0.0              0.0  ...   

   Tot size           IAT  Number   Magnitue      Radius    Covariance  \


In [ ]:
print(df.nunique())

flow_duration      2487676
Header_Length      1647793
Protocol Type         6333
Duration             13496
Rate               2785121
Srate              2785121
Drate                    1
fin_flag_number          2
syn_flag_number          2
rst_flag_number          2
psh_flag_number          2
ack_flag_number          2
ece_flag_number          1
cwr_flag_number          1
ack_count              325
syn_count             1080
fin_count              432
urg_count            19996
rst_count            68742
HTTP                     2
HTTPS                    2
DNS                      2
Telnet                   1
SMTP                     1
SSH                      1
IRC                      1
TCP                      2
UDP                      2
DHCP                     1
ARP                      2
ICMP                     2
IPv                      2
LLC                      2
Tot sum             336924
Min                  60233
Max                  81651
AVG                1636970
S

In [ ]:
#putting unique values columns into a list
one_unique_value = []
for column in df.columns:
  if df[column].nunique() == 1:
    one_unique_value.append(column)
print(one_unique_value)

['Drate', 'ece_flag_number', 'cwr_flag_number', 'Telnet', 'SMTP', 'SSH', 'IRC', 'DHCP']


In [ ]:
#deleting the unique value columns (Only one value is there in the columns)
for column in one_unique_value:
  df.drop(column, axis=1, inplace = True)
print(df.columns)

Index(['flow_duration', 'Header_Length', 'Protocol Type', 'Duration', 'Rate',
       'Srate', 'fin_flag_number', 'syn_flag_number', 'rst_flag_number',
       'psh_flag_number', 'ack_flag_number', 'ack_count', 'syn_count',
       'fin_count', 'urg_count', 'rst_count', 'HTTP', 'HTTPS', 'DNS', 'TCP',
       'UDP', 'ARP', 'ICMP', 'IPv', 'LLC', 'Tot sum', 'Min', 'Max', 'AVG',
       'Std', 'Tot size', 'IAT', 'Number', 'Magnitue', 'Radius', 'Covariance',
       'Variance', 'Weight', 'label', 'is_benign'],
      dtype='object')


In [ ]:
#dropping is_benign because it is in correlation to the label and may not be attribute
df.drop('is_benign', axis = 1, inplace = True)
print(df.columns)

Index(['flow_duration', 'Header_Length', 'Protocol Type', 'Duration', 'Rate',
       'Srate', 'fin_flag_number', 'syn_flag_number', 'rst_flag_number',
       'psh_flag_number', 'ack_flag_number', 'ack_count', 'syn_count',
       'fin_count', 'urg_count', 'rst_count', 'HTTP', 'HTTPS', 'DNS', 'TCP',
       'UDP', 'ARP', 'ICMP', 'IPv', 'LLC', 'Tot sum', 'Min', 'Max', 'AVG',
       'Std', 'Tot size', 'IAT', 'Number', 'Magnitue', 'Radius', 'Covariance',
       'Variance', 'Weight', 'label'],
      dtype='object')


In [ ]:
from sklearn.model_selection import train_test_split

# split the dataset
X = df.loc[:, [cols for cols in list(df.columns) if cols != "label"]]
Y = df.loc[:, "label"]
# train, test (dividing data into train and test data)
x_train, x_test, y_train, y_test = train_test_split(X, Y, test_size = 0.2,stratify=Y,random_state=42)


y_train.reset_index(drop = True, inplace =True)
y_test.reset_index(drop = True, inplace =True)

In [ ]:
print(" Shape of train data:")
print(x_train.shape)
print(y_train.shape)
print("\n Shape of test data:")
print(x_test.shape)
print(y_test.shape)

 Shape of train data:
(2985855, 38)
(2985855,)

 Shape of test data:
(746464, 38)
(746464,)


In [ ]:
from sklearn.preprocessing import StandardScaler

import pickle
print("Scaling")
stc = StandardScaler()
# Model fitting
stc.fit(x_train)
# Transform the training, validation, and test data using the same scaler
x_train_scaled= stc.transform(x_train)
x_test_scaled= stc.transform(x_test)
# Model saving
with open("/content/drive/MyDrive/1:1_Krish_Sathish/Model/cross_valid_scaler", "wb") as scaling_file:
    pickle.dump(stc, scaling_file)
    print("model saved")

Scaling
model saved


In [ ]:
#fetching feature names
features = X.columns
#converting array to dataframe (as label is in dataframe)
x_train_scaled = pd.DataFrame(x_train_scaled,columns=features)
x_test_scaled = pd.DataFrame(x_test_scaled,columns=features)

In [ ]:
#Check datatyoe of scaled data
type(x_train)

pandas.core.frame.DataFrame

In [ ]:
# Combining features and labels for train, validation and test data
# Train data
train_csv = pd.concat([x_train_scaled, y_train], axis = 1)
train_csv.reset_index(drop = True, inplace = True)
# Test data
test_csv = pd.concat([x_test_scaled, y_test], axis = 1)
test_csv.reset_index(drop = True, inplace = True)

In [ ]:
print(train_csv.head())

   flow_duration  Header_Length  Protocol Type  Duration      Rate     Srate  \
0       0.845634       2.472001      -1.040490  -0.43810 -0.081310 -0.081310   
1      -0.155719       0.552384      -0.603546  -0.43810 -0.070761 -0.070761   
2      -0.200352      -0.567503       1.075569  -0.38535 -0.081899 -0.081899   
3       1.382824      -0.486753      -1.199379  -0.51543 -0.081597 -0.081597   
4      -0.158543       1.653662      -0.603546  -0.43810 -0.057823 -0.057823   

   fin_flag_number  syn_flag_number  rst_flag_number  psh_flag_number  ...  \
0        -0.001157        -0.003472        -0.004292        -0.073011  ...   
1        -0.001157        -0.003472        -0.004292        -0.073011  ...   
2        -0.001157        -0.003472        -0.004292        -0.073011  ...   
3        -0.001157        -0.003472        -0.004292        -0.073011  ...   
4        -0.001157        -0.003472        -0.004292        -0.073011  ...   

        Std  Tot size       IAT    Number  Magnitu

In [15]:
train_csv.to_csv("/content/drive/MyDrive/1:1_Krish_Sathish/Data/processed_data_crossvalidation/train.csv")

In [ ]:
test_csv.to_csv("/content/drive/MyDrive/1:1_Krish_Sathish/Data/processed_data_crossvalidation/test.csv")